# 01 — WRDS Raw Data Extraction

Pulls four raw sources from WRDS and saves each as a `.parquet` file in `data/raw/`.
Run this notebook once; downstream notebooks load from those files.

| File | Source | Contents |
|---|---|---|
| `compustat_funda.parquet` | `comp.funda` | 18 financial line items + identifiers, 1999–2018, NYSE/NASDAQ |
| `compustat_bklabels.parquet` | `comp.company` | Delisting reason + date for bankruptcy labeling |
| `crsp_linktable.parquet` | `crsp.ccmxpf_linktable` | gvkey → permno mapping with validity dates |
| `crsp_dsf.parquet` | `crsp.dsf` | Daily returns, volume, price for linked universe |


In [1]:
!pip install -q wrds pyarrow


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [10]:
import os
import wrds
import pandas as pd

os.makedirs("data/raw", exist_ok=True)

db = wrds.Connection()  # reads ~/.pgpass after first login

Loading library list...
Done


## 1 · Compustat Annual Fundamentals (`comp.funda`)

Same 18 variables used in `00_build_master_dataset.ipynb`, plus the identifiers needed
to join with CRSP (`cik`, `tic`, `conm`, `sich`).  We pull `csho` and `prcc_f` separately
so `mktval` can be recomputed downstream; they are not saved as raw columns.

Filters mirror the master-dataset notebook:
- `indfmt='INDL'`, `datafmt='STD'`, `popsrc='D'`, `consol='C'`
- `exchg IN (11, 14)` — NYSE (11) and NASDAQ (14)
- `fyear` 1999–2018, `at > 0`

In [11]:
funda = db.raw_sql("""
    SELECT
        gvkey, cik, tic, conm, sich, exchg,
        fyear, datadate, fyr,
        -- fiscal year window (needed for CRSP temporal join downstream)
        DATE(DATE_TRUNC('month', datadate - INTERVAL '11 months')) AS begfyr,
        datadate AS endfyr,
        -- 18 core line items from the sowide/dataset paper
        act, at, cogs, dltt, dp, ebit, oibdp, gp,
        invt, lct, ni, re, rect, revt, lt, sale, xopr,
        -- mktval components (combined downstream)
        csho, prcc_f,
        -- additional balance-sheet items
        che,    -- cash & short-term investments
        icapt,  -- invested capital (total)
        ceq,    -- common equity
        dlc,    -- debt in current liabilities
        oancf,  -- net cash from operating activities
        capx,   -- capital expenditures
        txp,    -- taxes payable
        wcap    -- working capital
    FROM comp.funda
    WHERE indfmt = 'INDL'
      AND datafmt = 'STD'
      AND popsrc  = 'D'
      AND consol  = 'C'
      AND exchg  IN (11, 14)
      AND fyear  BETWEEN 1999 AND 2018
      AND at > 0
""", date_cols=["datadate", "begfyr", "endfyr"])

funda["mktval"] = funda["csho"] * funda["prcc_f"]
funda["gp"]    = funda["gp"].fillna(funda["sale"] - funda["cogs"])
funda["xopr"]  = funda["xopr"].fillna(funda["sale"] - funda["ebit"])
funda["fyear"] = funda["fyear"].astype(int)
funda = funda.sort_values(["gvkey", "fyear"]).reset_index(drop=True)

funda.to_parquet("data/raw/compustat_funda.parquet", index=False)
print(f"compustat_funda.parquet  →  {funda.shape}  |  {funda['gvkey'].nunique():,} companies")
funda.head(3)

compustat_funda.parquet  →  (96955, 39)  |  10,181 companies


,gvkey,cik,tic,conm,sich,exchg,fyear,datadate,fyr,begfyr,...,prcc_f,che,icapt,ceq,dlc,oancf,capx,txp,wcap,mktval
0,001004,0000001750,AIR,AAR CORP,5080,11,1999,2000-05-31,5,1999-06-01,...,13.875,1.241,519.962,339.515,26.314,10.051,22.344,3.027,347.451,372.751875
1,001004,0000001750,AIR,AAR CORP,5080,11,2000,2001-05-31,5,2000-06-01,...,14.0,13.809,520.199,340.212,13.652,46.093,13.134,2.059,360.464,377.118
2,001004,0000001750,AIR,AAR CORP,5080,11,2001,2002-05-31,5,2001-06-01,...,11.44,34.522,527.934,310.235,42.525,-33.315,12.112,3.847,286.192,364.5928


## 2 · Compustat Bankruptcy Labels (`comp.company`)

`dlrsn = '02'` flags Chapter 7/11 filings.  We capture `dldte` (delisting date) so the
label can be assigned to the fiscal year *before* the filing — consistent with the
master-dataset notebook.

In [12]:
bk = db.raw_sql("""
    SELECT
        gvkey,
        conm,
        dlrsn,
        dldte,
        EXTRACT(YEAR FROM dldte)::int - 1 AS failed_fyear
    FROM comp.company
    WHERE dlrsn = '02'
      AND dldte IS NOT NULL
      AND EXTRACT(YEAR FROM dldte) BETWEEN 2000 AND 2019
""", date_cols=["dldte"])

bk["failed_fyear"] = bk["failed_fyear"].astype(int)
bk.to_parquet("data/raw/compustat_bklabels.parquet", index=False)
print(f"compustat_bklabels.parquet  →  {bk.shape}  |  {bk['gvkey'].nunique():,} bankrupt companies")
bk.head(3)

compustat_bklabels.parquet  →  (263, 5)  |  263 bankrupt companies


,gvkey,conm,dlrsn,dldte,failed_fyear
0,001367,AMBER RESOURCES CO OF COLO,02,2012-08-31,2011
1,001703,APPLIED MAGNETICS CORP,02,2001-01-16,2000
2,002033,FAIRCHILD CORP -CL A,02,2011-11-01,2010


## 3 · CRSP–Compustat Link Table (`crsp.ccmxpf_lnkhist`)

We use **`ccmxpf_lnkhist`** (link history), the raw underlying table that WRDS's own
sample programs use ([CCM Link Collapse](https://wrds-www.wharton.upenn.edu/pages/wrds-research/macros/wrds-macros-ccm_lnktablesas/),
[CCM Merge](https://wrds-www.wharton.upenn.edu/pages/support/sample-programs/crsp/crsp-sample-programs-python/ccm-link-collapse/)).
`ccmxpf_linktable` is a pre-processed view; `lnkhist` is the authoritative source.

**Collapse step** (replicates WRDS's `ccm_lnktable.sas`): consecutive date ranges for
the same `gvkey/lpermno/lpermco` are merged into one record so there are no
spurious duplicates when joining to daily data.

Link types kept: `LC, LU, LD, LN, LS, LX` — the full set used in WRDS reference code.

In [13]:
import numpy as np

lnkhist = db.raw_sql("""
    SELECT gvkey, linkprim, liid, linktype,
           lpermno  AS permno,
           lpermco  AS permco,
           linkdt,
           linkenddt
    FROM crsp.ccmxpf_lnkhist
    WHERE linktype IN ('LC', 'LU', 'LD', 'LN', 'LS', 'LX')
    ORDER BY gvkey, lpermno, lpermco, linkdt, linkenddt
""", date_cols=["linkdt", "linkenddt"])

# ── Collapse consecutive date ranges (replicates ccm_lnktable.sas) ────────
# Within each gvkey/permno/permco group, if a record's linkdt touches or
# overlaps the previous record's linkenddt, pull linkdt back to the previous
# record's linkdt — merging the two rows into one contiguous window.
date_diff = 1
grp = ["gvkey", "permno", "permco"]

lnkhist["prev_linkdt"]     = lnkhist.groupby(grp)["linkdt"].shift()
lnkhist["prev_linkenddt"]  = lnkhist.groupby(grp)["linkenddt"].shift()

lnkhist["linkdt"] = np.where(
    (lnkhist["prev_linkenddt"].notna()) &
    (
        (lnkhist["linkdt"] == lnkhist["prev_linkenddt"]) |
        (lnkhist["linkdt"] == lnkhist["prev_linkenddt"] + pd.Timedelta(days=date_diff))
    ),
    lnkhist["prev_linkdt"],
    lnkhist["linkdt"],
)

lnkhist = lnkhist.drop(columns=["prev_linkdt", "prev_linkenddt"])

# Keep last record per gvkey/permno/linkdt after collapse
lnkhist = (
    lnkhist
    .sort_values(["gvkey", "permno", "linkdt"])
    .drop_duplicates(subset=["gvkey", "permno", "linkdt"], keep="last")
    .reset_index(drop=True)
)

# Sentinel for open-ended links (NULL linkenddt = still active)
lnkhist["linkenddt"] = lnkhist["linkenddt"].fillna(pd.Timestamp("2099-12-31"))

lnkhist.to_parquet("data/raw/crsp_linktable.parquet", index=False)
print(f"crsp_linktable.parquet  →  {lnkhist.shape}  |  {lnkhist['gvkey'].nunique():,} gvkeys  |  {lnkhist['permno'].nunique():,} permnos")
lnkhist.head(5)

crsp_linktable.parquet  →  (38738, 8)  |  36,147 gvkeys  |  36,938 permnos


,gvkey,linkprim,liid,linktype,permno,permco,linkdt,linkenddt
0,001000,P,01,LU,25881.0,23369.0,1970-11-13,1978-06-30
1,001001,P,01,LU,10015.0,6398.0,1983-09-20,1986-07-31
2,001002,C,01,LC,10023.0,22159.0,1972-12-14,1973-06-05
3,001003,C,01,LU,10031.0,6672.0,1983-12-07,1989-08-16
4,001004,P,01,LU,54594.0,20000.0,1972-04-24,2099-12-31


In [14]:
# Derive the permno universe from our funda gvkeys via the linktable
linktable_raw = pd.read_parquet("data/raw/crsp_linktable.parquet")
funda_raw     = pd.read_parquet("data/raw/compustat_funda.parquet")

universe_gvkeys = set(funda_raw["gvkey"].unique())
universe_permnos = (
    linktable_raw[linktable_raw["gvkey"].isin(universe_gvkeys)]["permno"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)
print(f"Universe: {len(universe_gvkeys):,} gvkeys  →  {len(universe_permnos):,} permnos")

Universe: 10,181 gvkeys  →  9,991 permnos


In [17]:
BATCH = 2_000
chunks = [universe_permnos[i:i+BATCH] for i in range(0, len(universe_permnos), BATCH)]

dsf_parts = []
for idx, batch in enumerate(chunks):
    permno_list = ",".join(str(p) for p in batch)
    chunk = db.raw_sql(f"""
        SELECT
            permno, date,
            ret, retx,
            ABS(prc)  AS prc,
            vol, shrout,
            cfacpr, cfacshr
        FROM crsp.dsf
        WHERE permno IN ({permno_list})
          AND date >= '2016-01-01'
    """, date_cols=["date"])
    dsf_parts.append(chunk)
    print(f"  batch {idx+1}/{len(chunks)}  →  {chunk.shape[0]:,} rows")

dsf = pd.concat(dsf_parts, ignore_index=True)
dsf = dsf.sort_values(["permno", "date"]).reset_index(drop=True)

dsf.to_parquet("data/raw/crsp_dsf.parquet", index=False)
print(f"\ncrsp_dsf.parquet  →  {dsf.shape}  |  {dsf['permno'].nunique():,} permnos")
dsf.head(3)

  batch 1/5  →  1,935,217 rows
  batch 2/5  →  2,296,959 rows
  batch 3/5  →  1,969,983 rows
  batch 4/5  →  1,194,564 rows
  batch 5/5  →  2,137,521 rows

crsp_dsf.parquet  →  (9534244, 9)  |  5,638 permnos


,permno,date,ret,retx,prc,vol,shrout,cfacpr,cfacshr
0,10025,2016-01-04,-0.051329,-0.051329,73.19,29246.0,5103.0,1.0,1.0
1,10025,2016-01-05,0.016532,0.016532,74.4,56220.0,5103.0,1.0,1.0
2,10025,2016-01-06,0.041532,0.041532,77.49,53924.0,5103.0,1.0,1.0


## 4 · CRSP Daily Stock File (`crsp.dsf`)

Raw daily rows — return, price, volume, shares — no transformation.

**Window: 2016-01-01 and Later**
Older daily data is largely superseded by Compustat annual fundamentals for earlier fyears.

The permno universe comes from the collapsed `ccmxpf_lnkhist` linked to our Compustat
gvkeys. When joining DSF to Compustat downstream, apply the linktable date validity:
```python
(linkdt <= date) & (date <= linkenddt)
```
as shown in the WRDS CCM merge reference.

In [18]:
FILES = [
    "data/raw/compustat_funda.parquet",
    "data/raw/compustat_bklabels.parquet",
    "data/raw/crsp_linktable.parquet",
    "data/raw/crsp_dsf.parquet",
]

for f in FILES:
    df = pd.read_parquet(f)
    size_mb = os.path.getsize(f) / 1e6
    print(f"{f:<45}  shape={str(df.shape):<18}  {size_mb:.1f} MB")

db.close()
print("\nAll extracts saved. WRDS connection closed.")

data/raw/compustat_funda.parquet               shape=(96955, 39)         16.3 MB
data/raw/compustat_bklabels.parquet            shape=(263, 5)            0.0 MB
data/raw/crsp_linktable.parquet                shape=(38738, 8)          1.0 MB
data/raw/crsp_dsf.parquet                      shape=(9534244, 9)        159.7 MB

All extracts saved. WRDS connection closed.
